# Working with OpenLLM in Pixeltable

Pixeltable's OpenLLM integration lets you call the AI subscriptions, provider API keys, and local
models you connect to OpenLLM through one OpenAI-compatible endpoint.

### Prerequisites

- An OpenLLM account with an API key (https://www.openllm.sh/sign-in)
- The OpenLLM daemon running on your machine (https://docs.openllm.sh/quickstart)

### Important notes

- Usage counts against the subscriptions and provider plans you connect to OpenLLM.
- Be mindful of sensitive data and consider security measures when integrating with external
  services.

First you'll need to install the required libraries and enter an OpenLLM API key. OpenLLM uses the
OpenAI SDK as its Python API, so we need to install it in addition to Pixeltable.

In [ ]:
%pip install -qU pixeltable openai

In [ ]:
import getpass
import os

if 'OPENLLM_API_KEY' not in os.environ:
    os.environ['OPENLLM_API_KEY'] = getpass.getpass('OpenLLM API Key:')

OpenLLM API Key: ········


By default, Pixeltable calls the local OpenLLM daemon at `http://127.0.0.1:8787/v1`. To use another
endpoint, set `OPENLLM_BASE_URL`.

Now let's create a Pixeltable directory to hold the tables for our demo.

In [1]:
import pixeltable as pxt

# Remove the 'openllm_demo' directory and its contents, if it exists
pxt.drop_dir('openllm_demo', force=True)
pxt.create_dir('openllm_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:***@/pixeltable?host=%2Fhome%2Fuser%2F.pixeltable%2Fpgdata
Created directory 'openllm_demo'.


## Chat completions

Create a table with an input column, then add a computed column that calls OpenLLM. The `plus`
model is a fallback alias: OpenLLM moves to the next model in the alias's chain when a provider is
rate-limited or unavailable. You can also use any model ID listed by the OpenLLM `models` endpoint.

In [2]:
from pixeltable.functions import openllm

t = pxt.create_table('openllm_demo/chat', {'input': pxt.String})

msgs = [{'role': 'user', 'content': t.input}]
t.add_computed_column(
    output=openllm.chat_completions(messages=msgs, model='plus')
)

Created table 'chat'.
Added 0 column values with 0 errors in 0.01 s


No rows affected.

In [3]:
# Parse the response into a new column
t.add_computed_column(response=t.output.choices[0].message.content)

Added 0 column values with 0 errors in 0.00 s


No rows affected.

In [4]:
# Start a conversation
t.insert(
    [
        {'input': 'What is the capital of France? Answer in one word.'},
        {'input': 'Explain what a computed column is in one sentence.'},
    ]
)
t.select(t.input, t.response).show()

Inserted 2 rows with 0 errors in 17.53 s (0.11 rows/s)


input,response
What is the capital of France? Answer in one word.,Paris
Explain what a computed column is in one sentence.,"A computed column is a table column whose value isn't stored directly but is derived automatically from an expression over other columns (or functions/constants) in the same row, either calculated on read or persisted on write depending on the database."


### Learn more

To learn more about advanced techniques like RAG operations in Pixeltable, check out the
[RAG Operations in Pixeltable](https://docs.pixeltable.com/howto/use-cases/rag-operations)
tutorial.